# Data Processing

Load the original model dataset, strip stale WTI/Brent features, download fresh prices from Yahoo Finance, define the target, and validate non-price columns against the raw source.

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

PROCESSED_DIR = Path("../..") / "data" / "processed"
RAW_DIR = Path("../..") / "data" / "raw"

df = pd.read_parquet(PROCESSED_DIR / "model_dataset_full.parquet")
df["Date"] = pd.to_datetime(df["Date"])
print(f"Loaded: {df.shape}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")

Loaded: (1958, 72)
Date range: 2016-10-18 to 2026-03-04


## Remove all Brent & WTI features

In [3]:
# Identify columns containing Brent or WTI (case-insensitive)
brent_wti_cols = [c for c in df.columns if any(k in c.lower() for k in ["brent", "wti"])]

# Also remove Target_Next_Day_Return
drop_cols = brent_wti_cols + ["Target_Next_Day_Return"]
removed = [c for c in drop_cols if c in df.columns]
kept = [c for c in df.columns if c not in removed]

print(f"REMOVED ({len(removed)}):")
for c in removed:
    print(f"  - {c}")

print(f"\nKEPT ({len(kept)}):")
for c in kept:
    print(f"  + {c}")

df = df[kept].copy()
print(f"\nDataset after removal: {df.shape}")

REMOVED (21):
  - WTI_Log_Returns
  - WTI_Ret_Lag1
  - WTI_Ret_Lag2
  - WTI_Ret_Lag3
  - WTI_Ret_Lag4
  - WTI_Ret_Lag5
  - WTI_Ret_Vol_5D
  - WTI_Ret_Vol_10D
  - WTI_Ret_Vol_20D
  - WTI_Ret_Lag6
  - WTI_Ret_Lag7
  - WTI_Ret_Lag8
  - WTI_Ret_Lag9
  - WTI_Ret_Lag10
  - WTI_Ret_Vol_60D
  - WTI_Spot_Momentum_5D
  - WTI_Spot_Momentum_20D
  - Brent_Log_Return
  - Brent_WTI_Spread
  - Brent_WTI_Spread_Chg
  - Target_Next_Day_Return

KEPT (51):
  + Date
  + Inv_WoW
  + Inv_WoW_Z4
  + Inv_WoW_Z13
  + AvgTone_DailyAvg
  + GoldsteinScale_DailyAvg
  + OVX_Log_Return
  + SPR_WoW
  + DaysSupply_WoW
  + Total_Stocks_WoW
  + Global_Tone_5D_Chg
  + Global_Goldstein_5D_Chg
  + Mentions_Volume_Surge
  + OPEC_Tension_Index
  + OPEC_Tension_5D_Chg
  + SAU_Goldstein_5D_Chg
  + SAU_Tone_5D_Chg
  + RUS_Goldstein_5D_Chg
  + RUS_Tone_5D_Chg
  + IRQ_Goldstein_5D_Chg
  + IRQ_Tone_5D_Chg
  + USA_Tone_5D_Chg
  + CHN_Tone_5D_Chg
  + FedAll_Qcexp
  + FedAll_Qolk
  + FedAll_Quncr
  + FedAll_Qemp
  + FedAll_Qsdtm
  + F

## Download fresh prices from Yahoo Finance

WTI, Brent, S&P 500, VIX, and DXY (US Dollar Index).

In [4]:
start = df["Date"].min().strftime("%Y-%m-%d")
end = (df["Date"].max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

tickers = {
    "CL=F":      "WTI_Close",
    "BZ=F":      "Brent_Close",
    "^GSPC":     "SP500_Close",
    "^VIX":      "VIX_Close",
    "DX-Y.NYB":  "DXY_Close",
}

price_dfs = []
for ticker, col_name in tickers.items():
    series = yf.download(ticker, start=start, end=end)["Close"].reset_index()
    series.columns = ["Date", col_name]
    series["Date"] = pd.to_datetime(series["Date"]).dt.tz_localize(None)
    price_dfs.append(series)
    print(f"{col_name:15s} {len(series)} rows  ({series['Date'].min().date()} to {series['Date'].max().date()})")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


WTI_Close       2357 rows  (2016-10-18 to 2026-03-04)
Brent_Close     2358 rows  (2016-10-18 to 2026-03-04)


[*********************100%***********************]  1 of 1 completed


SP500_Close     2356 rows  (2016-10-18 to 2026-03-04)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

VIX_Close       2356 rows  (2016-10-18 to 2026-03-04)
DXY_Close       2357 rows  (2016-10-18 to 2026-03-04)


## Merge prices and define target

In [5]:
for pdf in price_dfs:
    df = df.merge(pdf, on="Date", how="left")

# Define target: WTI log return at t+1 (next trading day)
df["Target"] = np.log(df["WTI_Close"] / df["WTI_Close"].shift(1)).shift(-1)

# Compute log returns for the new series
df["SP500_Log_Return"] = np.log(df["SP500_Close"] / df["SP500_Close"].shift(1))
df["VIX_Log_Return"] = np.log(df["VIX_Close"] / df["VIX_Close"].shift(1))
df["DXY_Log_Return"] = np.log(df["DXY_Close"] / df["DXY_Close"].shift(1))

# Reorder: Date, Target, price levels, log returns, then everything else
front_cols = ["Date", "Target", "WTI_Close", "Brent_Close", "SP500_Close", "VIX_Close", "DXY_Close",
              "SP500_Log_Return", "VIX_Log_Return", "DXY_Log_Return"]
other_cols = [c for c in df.columns if c not in front_cols]
df = df[front_cols + other_cols]

print(f"Shape: {df.shape}")
for c in front_cols:
    nans = df[c].isna().sum()
    if nans > 0:
        print(f"  {c} NaN: {nans}")
print(f"\nColumns ({len(df.columns)}):")
for c in df.columns:
    print(f"  {c}")

Shape: (1958, 60)
  Target NaN: 1
  SP500_Close NaN: 1
  VIX_Close NaN: 1
  DXY_Close NaN: 1
  SP500_Log_Return NaN: 3
  VIX_Log_Return NaN: 3
  DXY_Log_Return NaN: 3

Columns (60):
  Date
  Target
  WTI_Close
  Brent_Close
  SP500_Close
  VIX_Close
  DXY_Close
  SP500_Log_Return
  VIX_Log_Return
  DXY_Log_Return
  Inv_WoW
  Inv_WoW_Z4
  Inv_WoW_Z13
  AvgTone_DailyAvg
  GoldsteinScale_DailyAvg
  OVX_Log_Return
  SPR_WoW
  DaysSupply_WoW
  Total_Stocks_WoW
  Global_Tone_5D_Chg
  Global_Goldstein_5D_Chg
  Mentions_Volume_Surge
  OPEC_Tension_Index
  OPEC_Tension_5D_Chg
  SAU_Goldstein_5D_Chg
  SAU_Tone_5D_Chg
  RUS_Goldstein_5D_Chg
  RUS_Tone_5D_Chg
  IRQ_Goldstein_5D_Chg
  IRQ_Tone_5D_Chg
  USA_Tone_5D_Chg
  CHN_Tone_5D_Chg
  FedAll_Qcexp
  FedAll_Qolk
  FedAll_Quncr
  FedAll_Qemp
  FedAll_Qsdtm
  FedAll_Qwgs
  FedAll_Qbac
  FedEP_Qcexp
  FedEP_Qolk
  FedEP_Quncr
  FedEP_Qemp
  FedOG_Qolk
  FedOG_Quncr
  FedOG_Qcexp
  FedAll_Qcexp_5D_Chg
  FedAll_Qolk_5D_Chg
  FedAll_Quncr_5D_Chg
  FedA

## Sanity check: compare non-price columns against raw source

In [6]:
raw = pd.read_csv(RAW_DIR / "final_daily_data_enriched_12_countries.csv", parse_dates=["Date"])
print(f"Raw source: {raw.shape}")

skip_keywords = ["date", "target", "wti", "brent"]
check_cols = [c for c in df.columns if not any(k in c.lower() for k in skip_keywords)]

shared_cols = [c for c in check_cols if c in raw.columns]
only_in_model = [c for c in check_cols if c not in raw.columns]

print(f"\nColumns to check:    {len(check_cols)}")
print(f"Shared with raw:     {len(shared_cols)}")
print(f"Only in model (derived): {len(only_in_model)}")
if only_in_model:
    print(f"  {only_in_model}")

merged = df[["Date"] + shared_cols].merge(
    raw[["Date"] + shared_cols], on="Date", how="inner", suffixes=("_model", "_raw")
)
print(f"\nOverlapping dates: {len(merged)}")

print(f"\n{'Column':<35} {'MaxDiff':>10} {'MeanDiff':>10} {'Mismatches':>10}  Status")
print("-" * 85)

all_ok = True
for col in shared_cols:
    m_col = f"{col}_model"
    r_col = f"{col}_raw"
    diff = (merged[m_col] - merged[r_col]).abs()
    max_diff = diff.max()
    mean_diff = diff.mean()
    mismatches = (diff > 1e-6).sum()
    status = "OK" if mismatches == 0 else "MISMATCH"
    if mismatches > 0:
        all_ok = False
    print(f"  {col:<33} {max_diff:>10.6f} {mean_diff:>10.6f} {mismatches:>10}  {status}")

print(f"\n{'ALL COLUMNS MATCH RAW DATA' if all_ok else 'SOME COLUMNS DIFFER - investigate above'}")

Raw source: (3535, 216)

Columns to check:    56
Shared with raw:     16
Only in model (derived): 40
  ['SP500_Close', 'VIX_Close', 'DXY_Close', 'SP500_Log_Return', 'VIX_Log_Return', 'DXY_Log_Return', 'Inv_WoW', 'Inv_WoW_Z4', 'Inv_WoW_Z13', 'OVX_Log_Return', 'SPR_WoW', 'DaysSupply_WoW', 'Total_Stocks_WoW', 'Global_Tone_5D_Chg', 'Global_Goldstein_5D_Chg', 'Mentions_Volume_Surge', 'OPEC_Tension_Index', 'OPEC_Tension_5D_Chg', 'SAU_Goldstein_5D_Chg', 'SAU_Tone_5D_Chg', 'RUS_Goldstein_5D_Chg', 'RUS_Tone_5D_Chg', 'IRQ_Goldstein_5D_Chg', 'IRQ_Tone_5D_Chg', 'USA_Tone_5D_Chg', 'CHN_Tone_5D_Chg', 'FedAll_Qcexp_5D_Chg', 'FedAll_Qolk_5D_Chg', 'FedAll_Quncr_5D_Chg', 'FedAll_Qemp_5D_Chg', 'FedAll_Qsdtm_5D_Chg', 'FedAll_Qwgs_5D_Chg', 'FedAll_Qbac_5D_Chg', 'FedEP_Qcexp_5D_Chg', 'FedEP_Qolk_5D_Chg', 'FedEP_Quncr_5D_Chg', 'FedEP_Qemp_5D_Chg', 'FedOG_Qolk_5D_Chg', 'FedOG_Quncr_5D_Chg', 'FedOG_Qcexp_5D_Chg']

Overlapping dates: 1958

Column                                 MaxDiff   MeanDiff Mismatches  St

## Save processed data

Save the intermediate processed dataframe (fresh prices merged, target defined, no WTI/Brent-derived features yet). The feature engineering notebook picks up from here.

In [7]:
df.to_parquet(PROCESSED_DIR / "processed_base.parquet", index=False)
print(f"Saved processed_base.parquet: {df.shape}")

Saved processed_base.parquet: (1958, 60)
